# Pallid Swift and KNMI Data
Load the raw GBIF observations and KNMI daily weather records, then inspect the first 10 rows of each.

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path("data") / "raw"
gbif_path = data_dir / "0019564-260928105237408.csv"
knmi_path = data_dir / "result (1).txt"

gbif = pd.read_csv(gbif_path, sep="\t")
knmi = pd.read_csv(
    knmi_path,
    comment="#",
    header=None,
    names=["STN", "DATE", "DDVEC", "FG", "FHX", "TG", "TN", "TX", "RH", "PX"],
    skipinitialspace=True,
)

In [ ]:
gbif.head(10)

In [ ]:
knmi.head(10)

## Daily Pallid Swift Counts

Daily counts below cover KNMI's available date range. Dates with no GBIF records are included as zero, matching the daily weather calendar used for prediction.

In [ ]:
knmi_dates = pd.to_datetime(
    knmi["DATE"].astype("string"),
    format="%Y%m%d",
    errors="coerce",
)
event_dates = pd.to_datetime(
    gbif["eventDate"].astype("string").str[:10],
    errors="coerce",
).dt.normalize()

observed_daily_counts = event_dates.dropna().value_counts().sort_index()
weather_calendar = pd.date_range(knmi_dates.min(), knmi_dates.max(), freq="D")
daily_counts = (
    observed_daily_counts
    .reindex(weather_calendar, fill_value=0)
    .rename_axis("date")
    .rename("record_count")
    .to_frame()
)

top_20_dates = daily_counts.nlargest(20, "record_count").reset_index()
top_20_dates

In [ ]:
import math
import matplotlib.pyplot as plt

print("Descriptive statistics for daily Pallid Swift records:")
daily_counts["record_count"].describe().to_frame()

ax = daily_counts["record_count"].plot(figsize=(12, 5), linewidth=0.8)
ax.set_title("Daily Pallid Swift Records")
ax.set_xlabel("Date")
ax.set_ylabel("Number of records")
plt.tight_layout()
plt.show()

percentile_95 = daily_counts["record_count"].quantile(0.95)
suggested_threshold = max(1, math.ceil(percentile_95))
influx_days = daily_counts[
    daily_counts["record_count"] >= suggested_threshold
]

print(f"95th percentile: {percentile_95:.2f} records per day")
print(
    f"Suggested starting threshold: {suggested_threshold} or more records per day "
    "(at least 1)."
)
print(
    f"This flags {len(influx_days)} of {len(daily_counts)} days "
    f"({len(influx_days) / len(daily_counts):.1%})."
)
print("Since daily counts are discrete, ties can make the flagged share exceed 5%.")